# Multi-View 3D Line Reconstruction with LIMAP

This notebook demonstrates **full multi-view 3D line reconstruction** using
LIMAP's `line_triangulation` pipeline on the **ETH3D courtyard** scene
(38 images with COLMAP calibration).

We compare two detector configurations:
1. **LIMAP's built-in LSD** (`pytlsd`) — baseline
2. **Our native LSD** (`LsfmLimapDetector`) — integrated via `BaseDetector` subclass

**Pipeline:**
1. Load ETH3D scene with COLMAP calibration
2. Run LIMAP's full multi-view triangulation runner
3. Compare detectors: track counts, supporting views, track lengths
4. Visualize in 3D with Rerun

**Requirements:**
- [CVG LIMAP](https://github.com/cvg/limap) — **required**. Install with:
  `uv sync --extra limap` (see `docs/LIMAP.md` for details).
  **Do not** use `pip install limap` — the PyPI package is unrelated.
- ETH3D courtyard scene — download with `./tools/scripts/setup_eth3d.sh`

> For the simpler two-view stereo case, see `demo_stereo_reconstruction_limap.ipynb`
> or `demo_stereo_reconstruction.ipynb`.

In [ ]:
import sys
from pathlib import Path

# Discover workspace root
_nb = Path.cwd()
for _p in [_nb, *_nb.parents]:
    if (_p / 'MODULE.bazel').exists():
        WS = _p
        break
else:
    WS = _nb

# Ensure our Python packages are importable
for _d in ['python', 'bazel-bin/libs/geometry/python',
           'bazel-bin/libs/lsd/python', 'bazel-bin/libs/lfd/python',
           'bazel-bin/libs/edge/python', 'bazel-bin/libs/imgproc/python',
           'bazel-bin/libs/algorithm/python', 'bazel-bin/libs/eval/python']:
    p = str(WS / _d)
    if p not in sys.path:
        sys.path.insert(0, p)

import json
import numpy as np
import cv2
print(f'Workspace: {WS}')

## 2. Load ETH3D Courtyard Scene

The ETH3D courtyard scene has 38 DSLR images (6198x4129) with
COLMAP-calibrated camera poses. We load the scene metadata from
`cameras.json` and display sample images.

In [ ]:
from lsfm.data import TestImages
from lsfm.limap_compat import check_limap_available, LsfmLimapDetector
from lsfm.reconstruction import reconstruct_lines_multiview_full

check_limap_available()
print('LIMAP available')

ds = TestImages()
SCENE = 'courtyard'
scene_info = ds.eth3d_scene(SCENE)
print(f'Scene: {SCENE}')
print(f'  Scene dir:  {scene_info["scene_dir"]}')
print(f'  Images dir: {scene_info["images_dir"]}')
print(f'  COLMAP dir: {scene_info.get("colmap_dir", "N/A")}')

# Load camera metadata
with open(scene_info['cameras_json']) as f:
    meta = json.load(f)

n_images = meta['n_images']
frames = meta['frames']
print(f'  Images:     {n_images}')
print(f'  Resolution: {frames[0]["width"]}x{frames[0]["height"]}')

## 3. Visualize Scene — Sample Images & Camera Positions

In [ ]:
import matplotlib.pyplot as plt

# Show 4 sample images
sample_indices = [0, n_images // 4, n_images // 2, 3 * n_images // 4]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, idx in zip(axes, sample_indices):
    frame = frames[idx]
    img_path = scene_info['images_dir'] / frame['image']
    img = cv2.imread(str(img_path))
    if img is not None:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Image {idx}: {Path(frame["image"]).name}')
    ax.axis('off')
plt.suptitle(f'ETH3D {SCENE} — Sample Images ({n_images} total)', fontsize=14)
plt.tight_layout()
plt.show()

# Camera positions (t in cameras.json is COLMAP world-to-cam translation)
centers = []
for frame in frames:
    R = np.array(frame['R'])
    t = np.array(frame['t'])
    # Camera center = -R^T @ t
    center = -R.T @ t
    centers.append(center)
centers = np.array(centers)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(centers[:, 0], centers[:, 1], centers[:, 2], c='blue', s=30)
for i in sample_indices:
    ax.text(centers[i, 0], centers[i, 1], centers[i, 2], f'{i}', fontsize=8)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title(f'{SCENE} — Camera Positions')
plt.tight_layout()
plt.show()

## 4. Build LIMAP ImageCollection from COLMAP Data

We use LIMAP's `read_infos_colmap()` to load the undistorted COLMAP
model directly. This correctly handles COLMAP's pose convention
(`t = -R @ center`) without manual conversion.

**Important:** ETH3D images are 6198x4129 — we downscale to
`max_image_dim=1600` for practical runtime.

In [ ]:
import limap.pointsfm as pointsfm  # type: ignore[import-untyped]

# The COLMAP model is in dslr_calibration_undistorted/
colmap_path = str(scene_info['colmap_dir'].parent)
model_path = scene_info['colmap_dir'].name
image_path = 'images'

MAX_IMAGE_DIM = 1600

# read_infos_colmap returns (imagecols, neighbors, ranges)
imagecols, neighbors, ranges = pointsfm.read_infos_colmap(
    {},  # empty sfm config (uses defaults)
    colmap_path,
    model_path=model_path,
    image_path=image_path,
    n_neighbors=20,
)

print(f'ImageCollection: {imagecols.NumImages()} images')
print(f'Neighbors computed for {len(neighbors)} images')
print(f'Scene ranges: {ranges}')

# Show neighbor counts
neighbor_counts = [len(v) for v in neighbors.values()]
print(f'Neighbors per image: min={min(neighbor_counts)}, '
      f'max={max(neighbor_counts)}, mean={np.mean(neighbor_counts):.1f}')

## 5. Run LIMAP Pipeline — Built-in LSD Baseline

Run the full `line_triangulation` pipeline with LIMAP's built-in LSD
detector (`pytlsd`) as baseline.

In [ ]:
import time

print('Running LIMAP pipeline with built-in LSD...')
t_start = time.perf_counter()

result_builtin = reconstruct_lines_multiview_full(
    imagecols,
    neighbors=neighbors,
    ranges=ranges,
    max_image_dim=MAX_IMAGE_DIM,
    n_neighbors=20,
    n_visible_views=4,
    detector_class=None,  # LIMAP built-in LSD
)

t_builtin = time.perf_counter() - t_start
print(f'
Built-in LSD results:')
print(f'  Tracks:           {result_builtin["n_tracks"]}')
print(f'  Total time:       {t_builtin:.1f}s')
if result_builtin['supporting_views']:
    sv = result_builtin['supporting_views']
    print(f'  Supporting views: min={min(sv)}, median={np.median(sv):.0f}, max={max(sv)}')
if result_builtin['track_lengths']:
    tl = result_builtin['track_lengths']
    print(f'  Track lengths:    min={min(tl)}, median={np.median(tl):.0f}, max={max(tl)}')

## 6. Run LIMAP Pipeline — Our LSD Detector

Run the same pipeline with our native LSD detector (`LsfmLimapDetector`)
integrated via the `BaseDetector` subclass.

In [ ]:
print('Running LIMAP pipeline with our LSD detector...')
t_start = time.perf_counter()

result_ours = reconstruct_lines_multiview_full(
    imagecols,
    neighbors=neighbors,
    ranges=ranges,
    max_image_dim=MAX_IMAGE_DIM,
    n_neighbors=20,
    n_visible_views=4,
    detector_class='lsfm_lsd',  # Our native LSD
)

t_ours = time.perf_counter() - t_start
print(f'
Our LSD results:')
print(f'  Tracks:           {result_ours["n_tracks"]}')
print(f'  Total time:       {t_ours:.1f}s')
if result_ours['supporting_views']:
    sv = result_ours['supporting_views']
    print(f'  Supporting views: min={min(sv)}, median={np.median(sv):.0f}, max={max(sv)}')
if result_ours['track_lengths']:
    tl = result_ours['track_lengths']
    print(f'  Track lengths:    min={min(tl)}, median={np.median(tl):.0f}, max={max(tl)}')

## 7. Compare Results

In [ ]:
# Build comparison table
def _stats(result):
    sv = result['supporting_views']
    tl = result['track_lengths']
    return {
        'n_tracks': result['n_tracks'],
        'sv_min': min(sv) if sv else 0,
        'sv_median': float(np.median(sv)) if sv else 0,
        'sv_max': max(sv) if sv else 0,
        'tl_min': min(tl) if tl else 0,
        'tl_median': float(np.median(tl)) if tl else 0,
        'tl_max': max(tl) if tl else 0,
    }

s_builtin = _stats(result_builtin)
s_ours = _stats(result_ours)

print(f'{"Metric":<25s} {"Built-in LSD":>14s} {"Our LSD":>14s}')
print('-' * 55)
print(f'{"3D tracks":<25s} {s_builtin["n_tracks"]:>14d} {s_ours["n_tracks"]:>14d}')
print(f'{"Supporting views (min)":<25s} {s_builtin["sv_min"]:>14d} {s_ours["sv_min"]:>14d}')
print(f'{"Supporting views (med)":<25s} {s_builtin["sv_median"]:>14.0f} {s_ours["sv_median"]:>14.0f}')
print(f'{"Supporting views (max)":<25s} {s_builtin["sv_max"]:>14d} {s_ours["sv_max"]:>14d}')
print(f'{"Track length (min)":<25s} {s_builtin["tl_min"]:>14d} {s_ours["tl_min"]:>14d}')
print(f'{"Track length (med)":<25s} {s_builtin["tl_median"]:>14.0f} {s_ours["tl_median"]:>14.0f}')
print(f'{"Track length (max)":<25s} {s_builtin["tl_max"]:>14d} {s_ours["tl_max"]:>14d}')
print(f'{"Pipeline time (s)":<25s} {t_builtin:>14.1f} {t_ours:>14.1f}')

# Ratio
if s_builtin['n_tracks'] > 0:
    ratio = s_ours['n_tracks'] / s_builtin['n_tracks']
    print(f'
Our LSD produces {ratio:.2f}x the tracks of built-in LSD')

## 8. 3D Rerun Visualization

Visualize both reconstructions side by side with camera frustums.

In [ ]:
try:
    import rerun as rr
    import rerun.blueprint as rrb
    HAS_RERUN = True
except ImportError:
    HAS_RERUN = False
    print('Rerun not installed — skipping visualization')

if HAS_RERUN:
    from scipy.spatial.transform import Rotation as ScipyR

    rr.init('multiview_reconstruction_comparison', spawn=False)

    # Log camera frustums
    for idx, frame in enumerate(frames):
        K = np.array(frame['K'])
        R = np.array(frame['R'])
        t = np.array(frame['t'])
        w = frame['width']
        h = frame['height']

        # Camera-to-world: R_cw = R^T, t_cw = -R^T @ t (COLMAP convention)
        R_cw = R.T
        t_cw = -R_cw @ t
        quat_xyzw = ScipyR.from_matrix(R_cw).as_quat()

        rr.log(f'world/cameras/view_{idx}',
               rr.Pinhole(image_from_camera=K, width=w, height=h))
        rr.log(f'world/cameras/view_{idx}',
               rr.Transform3D(
                   rotation=rr.datatypes.Quaternion(xyzw=quat_xyzw),
                   translation=t_cw))

    # Log 3D lines — built-in LSD (blue)
    if result_builtin['linetracks']:
        pts_builtin = []
        for track in result_builtin['linetracks']:
            line = track.line
            pts_builtin.append([list(line.start), list(line.end)])
        rr.log('world/lines_builtin_lsd',
               rr.LineStrips3D(pts_builtin,
                               colors=[[50, 120, 255, 255]] * len(pts_builtin)))

    # Log 3D lines — our LSD (orange)
    if result_ours['linetracks']:
        pts_ours = []
        for track in result_ours['linetracks']:
            line = track.line
            pts_ours.append([list(line.start), list(line.end)])
        rr.log('world/lines_our_lsd',
               rr.LineStrips3D(pts_ours,
                               colors=[[255, 140, 40, 255]] * len(pts_ours)))

    # Blueprint: side-by-side 3D views
    bp = rrb.Blueprint(
        rrb.Horizontal(
            rrb.Spatial3DView(
                name='Built-in LSD',
                origin='world',
                contents=[
                    '+ world/cameras/**',
                    '+ world/lines_builtin_lsd',
                    '- world/lines_our_lsd',
                ],
            ),
            rrb.Spatial3DView(
                name='Our LSD',
                origin='world',
                contents=[
                    '+ world/cameras/**',
                    '+ world/lines_our_lsd',
                    '- world/lines_builtin_lsd',
                ],
            ),
        ),
    )

    rr.notebook_show(blueprint=bp)
    print(f'Rerun: {n_images} cameras, '
          f'{result_builtin["n_tracks"]} built-in tracks, '
          f'{result_ours["n_tracks"]} our tracks')

## 9. Statistics Summary

In [ ]:
# Distribution of supporting views
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if result_builtin['supporting_views']:
    axes[0].hist(result_builtin['supporting_views'],
                 bins=range(4, max(result_builtin['supporting_views']) + 2),
                 alpha=0.7, color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Supporting Views')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'Built-in LSD ({result_builtin["n_tracks"]} tracks)')

if result_ours['supporting_views']:
    axes[1].hist(result_ours['supporting_views'],
                 bins=range(4, max(result_ours['supporting_views']) + 2),
                 alpha=0.7, color='darkorange', edgecolor='black')
    axes[1].set_xlabel('Supporting Views')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Our LSD ({result_ours["n_tracks"]} tracks)')

plt.suptitle('Distribution of Supporting Views per Track', fontsize=14)
plt.tight_layout()
plt.show()

# Print timing breakdown
print('
Timing breakdown:')
for label, result in [('Built-in LSD', result_builtin), ('Our LSD', result_ours)]:
    print(f'  {label}:')
    for key, val in result['timings'].items():
        print(f'    {key}: {val:.1f}s')

## Summary

This notebook demonstrated **full multi-view 3D line reconstruction**
using LIMAP's `line_triangulation` runner:

| Feature | Details |
|---------|--------|
| **Scene** | ETH3D courtyard (38 images, 6198x4129) |
| **Calibration** | COLMAP undistorted model |
| **Pipeline** | LIMAP full: detection -> exhaustive matching -> multi-view triangulation -> filtering -> remerging |
| **Detectors** | Built-in LSD (pytlsd) vs. Our LSD (LsfmLimapDetector) |
| **Downscale** | max_image_dim=1600 for practical runtime |

### Key Takeaways
- `LsfmLimapDetector` integrates seamlessly with LIMAP's `BaseDetector` interface
- Full multi-view triangulation produces more robust tracks than pairwise stereo
- Track filtering (min visible views, reprojection) is critical for quality

### Next Steps
- Try other ETH3D scenes: `delivery_area`, `electro`
- Adjust `n_visible_views` threshold to trade off quantity vs. quality
- Enable bundle adjustment (`refinement.disable=False`) for higher accuracy
- See `demo_stereo_reconstruction_limap.ipynb` for the simpler stereo case